# Clasificación con Redes Neuronales (Breast Cancer)

## Objetivos
1. Cargar y preparar el dataset *Breast Cancer*.
2. Entrenar una red neuronal (MLP) con Keras/TensorFlow.
3. Evaluar desempeño con métricas (accuracy, ROC-AUC, reporte, matriz confusión).
4. Visualizar aprendizaje (curvas de pérdida/accuracy) y predicciones (probabilidades + ROC).
5. Guardar modelo y preprocesamiento (scaler) y reutilizarlos para inferencia.

### Notas
- Salida: probabilidad $P(y=1)$ (sigmoid)
- Pérdida: *binary cross-entropy*
- **Importante**: el escalado de variables suele ser clave para redes densas.


## Instalación (opcional)
si necesitas instalar dependencias, ejecuta:


In [1]:
# Si lo necesitas (en Colab):
# !pip -q install tensorflow scikit-learn matplotlib joblib

## Imports y configuración

In [2]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, accuracy_score
)
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


/Users/patricio/Documents/kibernum/.venv/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## Cargar dataset

In [3]:
data = load_breast_cancer()
X = data.data
y = data.target  # 0 = malignant, 1 = benign (en este dataset)

feature_names = data.feature_names
target_names = data.target_names

print("X shape:", X.shape)
print("Clases:", target_names, "-> y in {0,1}")
print("Proporción clase 1 (benign):", round(float(y.mean()), 3))


X shape: (569, 30)
Clases: ['malignant' 'benign'] -> y in {0,1}
Proporción clase 1 (benign): 0.627


## Split Train/Val/Test
Usamos `stratify=y` para conservar la proporción de clases en los splits.

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("y_train mean:", round(float(y_train.mean()), 3),
      "y_val mean:", round(float(y_val.mean()), 3),
      "y_test mean:", round(float(y_test.mean()), 3))


Train: (398, 30) Val: (85, 30) Test: (86, 30)
y_train mean: 0.628 y_val mean: 0.624 y_test mean: 0.628


## Escalado (StandardScaler)
**Regla de oro**: `fit` solo en train; `transform` en val/test para evitar fuga de información.

In [5]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print("Media (train, primera feature escalada):", round(float(X_train_s[:,0].mean()), 3))
print("Std  (train, primera feature escalada):", round(float(X_train_s[:,0].std()), 3))


Media (train, primera feature escalada): -0.0
Std  (train, primera feature escalada): 1.0


### Pregunta 1
- ¿Por qué el escalado suele ser crucial en redes neuronales densas?
- ¿Qué podría pasar si entrenas sin escalar en un dataset con variables en escalas muy distintas?


## Definir el modelo (MLP)
Arquitectura simple con `Dropout` para ayudar a reducir sobreajuste.

In [6]:
model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.25),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.15),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.AUC(name="auc")
    ]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,097 (16.00 KB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 0 (0.00 B)

## Entrenamiento con EarlyStopping
`EarlyStopping` corta el entrenamiento si `val_loss` no mejora y restaura los mejores pesos.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

print("Épocas entrenadas:", len(history.history["loss"]))
print("Mejor val_loss:", round(float(np.min(history.history["val_loss"])), 4))


## Curvas de aprendizaje
Gráfica de **pérdida** y **accuracy** para train y validación.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(history.history["loss"], label="train loss")
ax[0].plot(history.history["val_loss"], label="val loss")
ax[0].set_title("Pérdida (Binary Cross-Entropy)")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].legend()

ax[1].plot(history.history["accuracy"], label="train acc")
ax[1].plot(history.history["val_accuracy"], label="val acc")
ax[1].set_title("Accuracy")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].legend()

plt.tight_layout()
plt.show()


### Pregunta 2
- Observa las curvas train vs val: ¿ves señales de sobreajuste?
- Si hubiese sobreajuste, ¿qué 2 cambios probarías primero?
  - Ejemplos: más dropout, menos capas/neuronas, regularización L2, más datos, etc.


## Predicciones en TEST
El modelo devuelve probabilidades. Convertimos a clase con umbral 0.5.

In [ ]:
y_proba = model.predict(X_test_s, verbose=0).ravel()  # P(y=1)
y_pred = (y_proba >= 0.5).astype(int)

print("Ejemplo de probabilidades:", np.round(y_proba[:10], 3))
print("Ejemplo de clases predichas:", y_pred[:10])


## Evaluación en TEST
Incluye: Accuracy, ROC-AUC, reporte de clasificación y matriz de confusión.

In [ ]:
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("== Métricas en TEST ==")
print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC:  {auc:.4f}\n")

print("== Classification report ==")
print(classification_report(y_test, y_pred, target_names=target_names))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(values_format="d")
plt.title("Matriz de confusión (TEST)")
plt.show()


## Curva ROC

In [ ]:
fpr, tpr, thr = roc_curve(y_test, y_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Azar")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC (TEST)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Distribución de probabilidades predichas
Permite ver separación entre clases y cómo opera el umbral 0.5.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(y_proba[y_test == 0], bins=20, alpha=0.7, label="malignant (y=0)")
plt.hist(y_proba[y_test == 1], bins=20, alpha=0.7, label="benign (y=1)")
plt.axvline(0.5, linestyle="--", label="umbral 0.5")
plt.title("Distribución de P(y=1) por clase real (TEST)")
plt.xlabel("Probabilidad predicha de clase 1 (benign)")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()


## Ejemplos de predicciones individuales

In [ ]:
idx = np.random.choice(len(X_test_s), size=8, replace=False)
print("== Ejemplos de predicciones ==")
for i in idx:
    real = int(y_test[i])
    proba = float(y_proba[i])
    pred = int(y_pred[i])
    print(f"Real={real} ({target_names[real]}),  P(y=1)={proba:.3f},  Pred={pred} ({target_names[pred]})")


## Actividad: Ajustar el umbral
En problemas sensibles (salud), ajustar el umbral puede ser más importante que maximizar accuracy.

**Actividad 1**
- Cambia el umbral de decisión de 0.5 a 0.3 y 0.7.
- Compara matriz de confusión y métricas.
- ¿Qué umbral reduce más los falsos negativos?


In [ ]:
def evaluate_threshold(threshold: float):
    y_pred_thr = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred_thr)
    print(f"\n--- Umbral = {threshold} ---")
    print("Accuracy:", round(float(accuracy_score(y_test, y_pred_thr)), 4))
    print(classification_report(y_test, y_pred_thr, target_names=target_names))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    disp.plot(values_format="d")
    plt.title(f"Matriz de confusión (TEST) - umbral {threshold}")
    plt.show()

evaluate_threshold(0.3)
evaluate_threshold(0.5)
evaluate_threshold(0.7)


## Guardar modelo + scaler
Guardamos **ambos** para evitar mismatch en inferencia.

In [ ]:
MODEL_PATH = "breast_cancer_nn.keras"
SCALER_PATH = "breast_cancer_scaler.joblib"

model.save(MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)

print("Modelo guardado en:", MODEL_PATH)
print("Scaler guardado en:", SCALER_PATH)


## Reutilización posterior: cargar y predecir
Ejemplo de inferencia con nuevos datos (aquí usamos 5 filas del test como simulación).

In [ ]:
loaded_model = keras.models.load_model(MODEL_PATH)
loaded_scaler = joblib.load(SCALER_PATH)

X_new = X_test[:5]
X_new_s = loaded_scaler.transform(X_new)

new_proba = loaded_model.predict(X_new_s, verbose=0).ravel()
new_pred = (new_proba >= 0.5).astype(int)

print("== Reutilización (5 casos) ==")
for i in range(len(X_new)):
    print(f"Caso {i}: P(y=1)={new_proba[i]:.3f} -> Pred={new_pred[i]} ({target_names[new_pred[i]]})")


## Cierre y extensiones
**Checklist**
- [ ] Separaste train/val/test con `stratify`.
- [ ] Aplicaste `StandardScaler` (fit solo en train).
- [ ] Entrenaste con EarlyStopping.
- [ ] Evaluaste con ROC-AUC + matriz de confusión.
- [ ] Guardaste y cargaste modelo + scaler.

**Extensiones sugeridas**
1. Validación cruzada (K-Fold) para estimación más robusta.
2. Curva Precision-Recall (útil en desbalance).
3. Regularización L2 y comparación de arquitecturas.
4. Calibración de probabilidades (Platt/Isotonic).
